# Import Libraries and implement logger

In [220]:
## IMPORT LIBRARIES
import logging
import numpy as np
import pandas as pd
import torch

from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path
from io import StringIO
from datetime import datetime
from typing import Tuple, List, Optional, Dict
from torch import nn
from torch.utils.data import DataLoader, Dataset, TensorDataset
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold

In [221]:
# CREATE LOGGER FOR JUPYTER NOTEBOOKS
def setup_logger():
    # Create logger instance
    logger = logging.getLogger(__name__)
    logger.setLevel(logging.INFO)

    # Clear existing, avoid duplicate logs
    if logger.hasHandlers():
        logger.handlers.clear()
    
    # Create FileHandler in overrite mode
    file_handler = logging.FileHandler('../../logmodeler.log', mode ='w')

    # Set format
    formatter = logging.Formatter('%(asctime)s - %(levelno)s - %(lineno)d - %(module)s - %(message)s')
    file_handler.setFormatter(formatter)

    # Add FileHandler to logger
    logger.addHandler(file_handler)

    return logger

logger = setup_logger()

# Inspcet the data read in after the Visualzer

In [222]:
## Define function for inspection
def inspect_raw_data(df, output_path=None):
    '''
    Inspect a DataFrame and save info to a text file.
    Parameters:
    df : pandas DataFrame
    output_path : Path or str, optional
        Where to save the output file. If None and save_to_file is True,
        will use current directory
    '''

    # Set display options
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    
    # Prepare the information
    info_text = []
    info_text.append(f"Generated on: {datetime.now()}")
    info_text.append(f"Shape: {df.shape}")
    
    # Get info in string format
    buffer = StringIO()
    df.info(buf=buffer)
    info_text.append(f"\nDataFrame Info:")
    info_text.append(buffer.getvalue())

    # Add first 5 rows
    buffer_head = StringIO()
    df.head().to_string(buf=buffer_head)
    info_text.append(f"\nFirst 5 rows:")
    info_text.append(buffer_head.getvalue())

    # Add last 5 rows
    buffer_tail = StringIO()
    df.tail().to_string(buf=buffer_tail)
    info_text.append(f"\nLast 5 rows:")
    info_text.append(buffer_tail.getvalue())
    
    # Join all information
    full_report = '\n'.join(info_text)
    output_path = Path(output_path)
    output_path.write_text(full_report)
    logger.info(f"Report saved to: {output_path}")
    
    return full_report

In [223]:
# READ IN
data_path_eso = Path('..') / '..' / 'data' / 'processed' / 'Visualizer' / 'eso'
data_path_ger = Path('..') / '..' / 'data' / 'processed' / 'Visualizer' / 'ger'

# ESO
# Normalized
X_train_eso_scaled = pd.read_csv(data_path_eso / 'X_train_eso_scaled.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']
X_test_eso_scaled = pd.read_csv(data_path_eso / 'X_test_eso_scaled.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']
# Not normalized
y_train_eso = pd.read_csv(data_path_eso / 'y_train_eso.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']
y_test_eso = pd.read_csv(data_path_eso / 'y_test_eso.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']

# GER
# Normalized
X_train_ger_scaled = pd.read_csv(data_path_ger / 'X_train_ger_scaled.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']
X_test_ger_scaled = pd.read_csv(data_path_ger / 'X_test_ger_scaled.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']
# Not normalized
y_train_ger = pd.read_csv(data_path_ger / 'y_train_ger.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']
y_test_ger = pd.read_csv(data_path_ger / 'y_test_ger.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']

In [224]:
"""## SAVE FORE INSPECTION
# ESO
# Normalized
eso_scaled_Xtrain = 'X_train_eso_scaled'; eso_scaled_Xtest = 'X_test_eso_scaled'
_ = inspect_raw_data(X_train_eso_scaled, output_path=Path(f'{data_path}/../Modeler/eso/{eso_scaled_Xtrain}.txt'))
_ = inspect_raw_data(X_test_eso_scaled, output_path=Path(f'{data_path}/../Modeler/eso/{eso_scaled_Xtest}.txt'))
# Not normalized
eso_ytrain = 'y_train_eso'; eso_ytest = 'y_test_eso'
_ = inspect_raw_data(y_train_eso, output_path=Path(f'{data_path}/../Modeler/eso/{eso_ytrain}.txt'))
_ = inspect_raw_data(y_test_eso, output_path=Path(f'{data_path}/../Modeler/eso/{eso_ytest}.txt'))
# GER
# Normalized
ger_scaled_Xtrain = 'X_train_ger_scaled'; ger_scaled_Xtest = 'X_test_ger_scaled'
_ = inspect_raw_data(X_train_ger_scaled, output_path=Path(f'{data_path}/../Modeler/ger/{ger_scaled_Xtrain}.txt'))
_ = inspect_raw_data(X_test_ger_scaled, output_path=Path(f'{data_path}/../Modeler/ger/{ger_scaled_Xtest}.txt'))
# Not normalized
ger_ytrain = 'y_train_ger'; ger_ytest = 'y_test_ger'
_ = inspect_raw_data(y_train_ger, output_path=Path(f'{data_path}/../Modeler/ger/{ger_ytrain}.txt'))
_ = inspect_raw_data(y_test_ger, output_path=Path(f'{data_path}/../Modeler/ger/{ger_ytest}.txt'))"""

"## SAVE FORE INSPECTION\n# ESO\n# Normalized\neso_scaled_Xtrain = 'X_train_eso_scaled'; eso_scaled_Xtest = 'X_test_eso_scaled'\n_ = inspect_raw_data(X_train_eso_scaled, output_path=Path(f'{data_path}/../Modeler/eso/{eso_scaled_Xtrain}.txt'))\n_ = inspect_raw_data(X_test_eso_scaled, output_path=Path(f'{data_path}/../Modeler/eso/{eso_scaled_Xtest}.txt'))\n# Not normalized\neso_ytrain = 'y_train_eso'; eso_ytest = 'y_test_eso'\n_ = inspect_raw_data(y_train_eso, output_path=Path(f'{data_path}/../Modeler/eso/{eso_ytrain}.txt'))\n_ = inspect_raw_data(y_test_eso, output_path=Path(f'{data_path}/../Modeler/eso/{eso_ytest}.txt'))\n# GER\n# Normalized\nger_scaled_Xtrain = 'X_train_ger_scaled'; ger_scaled_Xtest = 'X_test_ger_scaled'\n_ = inspect_raw_data(X_train_ger_scaled, output_path=Path(f'{data_path}/../Modeler/ger/{ger_scaled_Xtrain}.txt'))\n_ = inspect_raw_data(X_test_ger_scaled, output_path=Path(f'{data_path}/../Modeler/ger/{ger_scaled_Xtest}.txt'))\n# Not normalized\nger_ytrain = 'y_train_

# Begin modeling after read in the dataframes 
#### Set requirements for building a predictor 
    - Use pytorch LSTM
      Consider: https://pytorch.org/tutorials/beginner/nlp/sequence_models_tutorial.html, https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html
    - Use dataclasses, where the model is build, name it Modeler 
    - Consider the figure of the modeling process
    - Use the already split and normalized dataframes X_train_XXX_scaled, X_test_XXX_scaled, y_train_XXX, y_test_XXX
    - Consider the information in the text files and that there is also are 8 files all over, XXX is replaced with eso or ger
    - Use cross Validation
      Consider: https://scikit-learn.org/1.5/modules/cross_validation.html
    - Use proper Evaluation Metric
    - Consider: batch size, epoche, learning rate, learning rate dk

In [225]:
## SET FIXED VARIABLES FOR ESO AND GER
class DatasetType(Enum):
    ESO = 'eso'
    GER = 'ger'

In [226]:
## CONFIGURATIONS FOR THE MODEL
@dataclass
class ModelConfig:
    # INITIALIZATION 
    dataset_type: DatasetType # Declaration of used type of dataset
    input_size: int = field(init=False) # Number of expected features in the input x
    hidden_size: int = 256 # Number of features in the hidden state h
    num_layers: int = 3 # Number of recurrent layers (with 2, 2th would output from 1st)
    
    #bias: True # Use bias weights b_ih and b_hh (default:True)
    #batch_first: True # I/O tensors provided as (batch, seq, feature for True) and (seq, batch, feature for False) 
    dropout: float = 0.1 # introduces a dropout layer on the output (expect last layer) (default=0)
    #bidirectional: False # (default=False)
    #proj_size: int = 0 #  if>0, will use LSTM with projections of corresponding size. (default=0)

    output_size: int = 1 # Size of output
    batch_size: int = 64 # Number of samples processed together in one forward/backward pass during training.
    epochs: int = 40 # Number of epochs
    learning_rate: float = 0.001 # step size directed gradient
    lr_decay: float = 0.98 
    sequence_length: int = 24 # Number of steps to look back
    n_splits: int = 2 # Number of folds for cross validation

    def __post_init__(self):
        self.input_size = 85 if self.dataset_type == DatasetType.ESO else 66 # Changed from 86/67 to 85/66 because we will loose ID column in modeling


In [227]:
## DEFINE SEQUENCE
class EnergyDataset(Dataset):
    # Transform to numpy for numerical or tensor-based operations
    def __init__(self, X:np.ndarray, y:np.ndarray, sequence_length: int):

        # Pad the input sequence to maintain the same length as original data
        padding = np.zeros((sequence_length - 1, X.shape[1]))
        X_padded = np.vstack([padding, X])
        self.X = torch.FloatTensor(X_padded)
        self.y = torch.FloatTensor(y)

        #self.X = torch.FloatTensor(X)
        #self.y = torch.FloatTensor(y)
        self.sequence_length = sequence_length

    # Get sequence length
    def __len__(self):

        return len(self.y) 
        #return len(self.X) - self.sequence_length + 1 
    
    # 
    def __getitem__(self, idx):
        x_sequence = self.X[idx:idx + self.sequence_length] # Input sequence (sequence_length steps starting at idx)
        
        y_target = self.y[idx]
        #y_target = self.y[idx + self.sequence_length - 1] # Target value (last step of the sequence)
        
        '''
        self.X[0:0 + 24] extracts the first 24 elements (idx 0...23) of self.X
        self.y[0 + 24 - 1] extracts the 24th element (idx 23) of self.y
        '''

        logger.info(f'Shape and type of x_sequence: {x_sequence.shape}, {type(x_sequence)}')
        logger.info(f'Shape and type of y_target: {y_target.shape}, {type(y_target)}')

        return x_sequence, y_target

In [228]:
class LSTMModel(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__() # Call method from a parent (super) class
        self.hidden_size = config.hidden_size
        self.num_layers = config.num_layers
        self.lstm = nn.LSTM(
            input_size=config.input_size,
            hidden_size=config.hidden_size,
            num_layers=config.num_layers,
            batch_first=True,
            dropout=config.dropout
        )
        #self.fc = nn.Linear(config.hidden_size, config.output_size) # Linear only for final prediction
        # Added intermediate layers
        self.fc1 = nn.Linear(config.hidden_size, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(config.dropout)
        self.fc2 = nn.Linear(128, config.output_size)

    def init_hidden(self, batch_size, device):
        return (torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device),
                torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device))
    
    def forward(self, x, hidden=None):

        if hidden is None:
            hidden = self.init_hidden(x.size(0), x.device)

        lstm_out, hidden = self.lstm(x, hidden) # Pass input through LSTM layer
        #predictions = self.fc(lstm_out[:, -1, :]) # Pass the last time steps output through the fully connected layer
        out = self.fc1(lstm_out[:, -1, :])
        out = self.relu(out)
        out = self.dropout(out)
        predictions = self.fc2(out)
        return predictions, hidden        

self.fc = nn.Linear(config.hidden_size, config.output_size) \
$\rightarrow$ Defines a fully connected (fc) layer. 
Creates a linear transformation of the input features to the output features: 
$y=x~W^T+b$

* $x$ is the input tensor of size (batch_size, in_features)
* $W$ is the weight matrix of shape (out_features, in_features)
* $b$ is the bias vector of shape (out_features)
* $y$ is the output tensor of shape (batch_size, out_features)

config.hidden_size: The number of input features to the linear layer. After passing through the LSTM layer, the output will have a shape (batch_size, sequence_length, hidden_size) \
$\rightarrow$ https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html batch_first: True $\rightarrow$ (batch, seq, feature))

config.output_size: The number of output features of the linear layer. Represents the final dimensionality of the model output. (Typically 1 for regression)

---

$x$: input tensor with shape (batch_size, sequence_length)
* batch_size: Number of sequences in a batch. E.g. dataset with 10 000 samples: 10 000/batch_size
* sequence_length: Number of time steps in each sequence.
* input_size: Number of features at each time step

lstm_out: output features for each time step, with shape (batch_size, sequence_length, hidden_size) contains output feature ($h_t$: hidden state vector for all time steps $t$ in the sequence) \
_: The hidden and cell states. Tupel of ($h_n$: hidden state of the last time step for all layers, $c_n$: cell state of the last time step for all layers) not used here, so we discard them with _  

lstm_out[:, -1, :]: Selects the hidden state output for the last time step of each sequence
* :: Selects all batches.
* -1: Selects the last time step (index -1 refers to the last element in the sequence).
* :: Selects all features (of size hidden_size) for that time step.

$\rightarrow$ Get tensor with shape (batch_size, hidden_size)

---

1. The LSTM processes the sequence (x) with shape (batch_size, sequence_length, input_size). 
2. The output tensor (lstm_out) retains the batch_first shape, i.e., (batch_size, sequence_length, hidden_size).  
3. You extract the last time step's output for each sequence, resulting in (batch_size, hidden_size). 
4. The fully connected (self.fc) layer transforms this to (batch_size, output_size). 


In [ ]:
@dataclass
class Modeler:
    
    # Define attributes
    config: ModelConfig # Get configurations
    device: torch.device = field(default_factory=lambda: torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
    model: Optional[LSTMModel] = None # Declare model attribute, LSTMModel for training
    criterion: Optional[nn.Module] = None # For reference a loss function module (as CrossEntrupyLoss, MSELoss)
    optimizer: Optional[torch.optim.Optimizer] = None # For referencing an optimizer 
    #scheduler: Optional[torch.optim.lr_scheduler.StepLR] = None # For referencing a learning rate
    scheduler: Optional[torch.optim.lr_scheduler.StepLR] = None
    '''
    With the device:
    field function is capable to specify default values and meta data for class attributes
    default_factory argument provides a function that returns default values. With lambda function it checks, if a Compute Unified Device Architecture (CUDA) device is available.
    CUDA is a Application Programming Interface (API).
    In this case it refers to the GPU. If GPU is not available, it selects the CPU if not.
    '''

    # Method for initalizing model, loss function, optimizer, scheduler ----------------------------------------------------------------
    def __post_init__(self):
        self.model = LSTMModel(self.config).to(self.device) # Instance of LSTMModel passing config as an argument to device
        self.criterion = nn.MSELoss() # Set MSELoss, common for regression
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.config.learning_rate, weight_decay=0.01) # Init. adam optimizer to adapt learning rate for each parameter (lr is initial learning rate)
        #self.scheduler = torch.optim.lr_scheduler.ExponentialLR(self.optimizer, gamma=self.config.lr_decay) 
        self.scheduler = torch.optim.lr_scheduler.StepLR(
            self.optimizer,
            step_size=5,  # Reduce LR every 5 epochs
            gamma=self.config.lr_decay
        )
        '''
        Init. learning rate scheduler type StepLR. 
        This scheduler reduces the learning rate by a factir if ganna for every step_size epochs. 
        The step_size and gamma parameters are taken from the config object.
        '''
        
    # Method to drop ID, converting to numpy array, reshaping target ----------------------------------------------------------------
    def prepare_data(self, X: pd.DataFrame, y: pd.DataFrame) -> Tuple[DataLoader, np.array]:
        # Remove ID column with datetime and convert to numpy array
        X_values = X.drop('ID', axis=1).values
        y_values = y.drop('ID', axis=1).values.reshape(-1,1) # Transform y_values into 2D array with a single column

        # Create instance of EnergyDataset
        dataset = EnergyDataset(X_values, y_values, self.config.sequence_length)
        # Create instance of DataLoader that handles batching and shuffling
        dataloader = DataLoader(dataset, batch_size=self.config.batch_size, shuffle=True)

        # Access the first batch to get an example
        logger.info(f'Batch size of dataloader: {dataloader.batch_size}')
        logger.info(f'Dataset size of dataloader: {len(dataloader.dataset)}')
        data, target = next(iter(dataloader))
        logger.info(f'Shape and type of data: {data.shape}, {type(data)}')
        logger.info(f'Shape and type of target: {target.shape}, {type(target)}')

        return dataloader, y_values
    
    # Method to train model for one epoche ----------------------------------------------------------------
    def train_epoch(self, train_loader: DataLoader) -> float:
        self.model.train() # Set the initialized LSTMModel to training mode
        total_loss = 0 # Initialize a total_loss variable to accumulate the loss across all batches
        
        # Initialize hidden state
        hidden = None
        hidden = self.model.init_hidden(train_loader.batch_size, self.device)

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(self.device)
            batch_y = batch_y.to(self.device) # For processing, move batch tensor and target to device 

            self.optimizer.zero_grad()

            #self.optimizer.zero_grad() # Zero the gradients of the optimizer
            predictions, hidden = self.model(batch_X, hidden) # Use the model training mode to make predictions with the batch_X

            hidden = tuple(h.detach() for h in hidden) # Detach hidden states
            loss = self.criterion(predictions, batch_y) # Calculate the loss with the selected criterion

            loss.backward() # Backpropagation for the loss

            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

            self.optimizer.step() # Step method to update the model parameters based on the calculated gradients in the backpropagation

            total_loss = total_loss + loss.item() # Accumulate the loss of the batch

        return total_loss / len(train_loader) # Return average loss of the training epoche
        
    # Method to validate the models performance on a validation dataset ----------------------------------------------------------------
    def validate(self, val_loader: DataLoader) -> Tuple[float, np.ndarray, np.ndarray]:
        self.model.eval() # Set the initialized LSTMModel to evaluation mode
        total_loss = 0 # Initialize a total_loss variable to accumulate the loss across all batches
        predictions = [] # Initialize list to store model predictions for validation samples (y_pred)
        actuals = [] # Initialize list to store actual target values for validation samples (y_test)

        # Initialize hidden state
        hidden = self.model.init_hidden(val_loader.batch_size, self.device)
        
        with torch.no_grad(): 
            ''' 
            In the validation loop, the gradients should not be updated. 
            Therefore, ensure that the gradients are not calculated during the forward pass of the valudation.
            To implement this, use torch.no_grad 
            '''
            for batch_X, batch_y in val_loader:
                batch_X = batch_X.to(self.device)
                batch_y = batch_y.to(self.device) # For processing, move batch tensor and target to device

                batch_predictions, hidden = self.model(batch_X, hidden) # Use the model validation mode to make predictions with the batch_X
                # Don't need to detach hidden here since we're in no_grad context

                loss = self.criterion(batch_predictions, batch_y) # Calculate the loss with the selected criterion
                total_loss = total_loss + loss.item() # Accumulate the loss of the batch

                predictions.extend(batch_predictions.cpu().numpy()) # Append current batch predictions (as numpy array from device) to prediction list
                actuals.extend(batch_y.cpu().numpy()) # Append current batch actual target values (as numpy array from device) to prediction list

            # Return average loss of the validation epoche, an array with predictions, an array with targets
            return total_loss / len(val_loader), np.array(predictions), np.array(actuals) 
    
    # Method for cross validation with scikit-learn ----------------------------------------------------------------
    def cross_validate(self, X: pd.DataFrame, y: pd.DataFrame) -> dict:
        kfold = KFold(n_splits=self.config.n_splits, shuffle=True, random_state=42) # Create cross validator with specified splits shuffling the data for each split
        # Create dictionay to store the evaluation metrics
        cv_scores = {
            'mse': [],
            'mae': [],
            'r2': [],
            'train_losses': [],
            'val_losses': []
        }

        # Remove ID column with datetime and convert to numpy array
        X_values = X.drop('ID', axis = 1).values
        # Scale target values
        
        y_values = y['FOSSIL'].values           

        # Divide the train data into training and validation set for each fold using kfold.split. Than for each fold ...
        for fold, (train_idx, val_idx) in enumerate(kfold.split(X_values)):
            print(f'Fold {fold + 1}/{self.config.n_splits}')

            # ... reset model, optimizer, scheduler 
            self.__post_init__() 

            # ... extract train and validation data based on the indicies
            X_train, X_val = X_values[train_idx], X_values[val_idx]
            y_train, y_val = y_values[train_idx], y_values[val_idx]

            # Add padding for sequences
            padding = np.zeros((self.config.sequence_length - 1, X_train.shape[1]))
            X_train_padded = np.vstack([padding, X_train])
            X_val_padded = np.vstack([padding, X_val])
            train_dataset = EnergyDataset(X_train_padded, y_train.reshape(-1, 1), self.config.sequence_length)
            val_dataset = EnergyDataset(X_val_padded, y_val.reshape(-1, 1), self.config.sequence_length)

            # ... create EnergyDataset instance for training and validation set. 
            # Reshape because EnergyDataset expects target values to be in 2D array even if it is a single-dimensional value
            #train_dataset = EnergyDataset(X_train, y_train.reshape(-1, 1), self.config.sequence_length)
            #val_dataset = EnergyDataset(X_val, y_val.reshape(-1, 1), self.config.sequence_length)

            # ... create DataLoader instance for training and validation set. 
            train_loader = DataLoader(train_dataset, batch_size=self.config.batch_size, shuffle=True, drop_last=True)
            val_loader = DataLoader(val_dataset, batch_size=self.config.batch_size, drop_last=True)

            '''
            Example case: dataset with 1000 data points and one target variable
            Without reshaping, y_values is a 1D array with shape (1000, 0). 
            Reshaping it to (1000, 1) creates a 2D array where each row represents a single datapoint and each column represents the target values.
            '''

            best_val_loss = float('inf')
            patience = 5
            patience_counter = 0

            # ... create empty list for train and validation losses
            train_losses = []
            val_losses = []

            # ... iterator over the set number of epochs
            for epoch in range(self.config.epochs):
                train_loss = self.train_epoch(train_loader) # Call train_epoch method
                val_loss, predictions, actuals = self.validate(val_loader) # Call validate method

                # Append results of the train_epoch and validate method
                train_losses.append(train_loss) 
                val_losses.append(val_loss)

                # Update the learning rate according to the specified learning rate schedule
                self.scheduler.step(val_loss)

                # Check if the current (epoch + 1) is divisible by 10 and print if it is
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    patience_counter = 0
                else:
                    patience_counter += 1
                    
                if patience_counter >= patience:
                    logger.info(f'Early stopping at epoch {epoch + 1}')
                    break
                    
                if (epoch + 1) % 5 == 0:
                    logger.info(f'Epoch {epoch + 1}/{self.config.epochs}')
                    logger.info(f'Train loss: {train_loss:.4f}')
                    logger.info(f'Val loss: {val_loss:.4f}')
                

            # Calculate metrics
            mse = mean_squared_error(actuals, predictions)
            mae = mean_absolute_error(actuals, predictions)
            r2 = r2_score(actuals, predictions)

            # Append metrics
            cv_scores['mse'].append(mse)
            cv_scores['mae'].append(mae)
            cv_scores['r2'].append(r2)
            cv_scores['train_losses'].append(train_losses)
            cv_scores['val_losses'].append(val_losses)

            # Print fold results
            logger.info(f"\nFold {fold + 1} Results:")
            logger.info(f"MSE: {mse:.2f}")
            logger.info(f"MAE: {mae:.2f}")
            logger.info(f"R2: {r2:.4f}")

        # Calculate and print average scores
        print(f"\nCross-validation results for {self.config.dataset_type.value}:")
        print(f"Average MSE: {np.mean(cv_scores['mse']):.2f} ± {np.std(cv_scores['mse']):.2f}")
        print(f"Average MAE: {np.mean(cv_scores['mae']):.2f} ± {np.std(cv_scores['mae']):.2f}")
        print(f"Average R2: {np.mean(cv_scores['r2']):.4f} ± {np.std(cv_scores['r2']):.4f}")

        return cv_scores
        
    # Method to predict ----------------------------------------------------------------
    def predict(self, X_test: pd.DataFrame) -> np.ndarray:
        self.model.eval()  # Set the initialized LSTMModel to evaluation mode

        # Prepare test data with padding
        X_values = X_test.drop('ID', axis=1).values
        padding = np.zeros((self.config.sequence_length - 1, X_values.shape[1]))
        X_padded = np.vstack([padding, X_values])


        # Create instance of EnergyDataset specified for the test data. 
        test_dataset = EnergyDataset(
            #X_test.drop('ID', axis=1).values, 
            X_padded,
            np.zeros((len(X_test), 1)),
            self.config.sequence_length
        )
        '''
        Drop the 'ID' column.
        Create an array of zeros with the same length as the test data but with a single column. 
        This is needed because the original data have target values and EnergyDataset expects features and targets.
        This provides dummy zero targets since for the prediction only the feature is needed.
        '''

        # Create instance of DataLoader specified for the test data.
        test_loader = DataLoader(
            test_dataset,
            batch_size=self.config.batch_size,
            shuffle=False,
            drop_last=False
        )

        # Initialize list to store the predictions
        predictions = []
        # Initialize hidden state
        hidden = self.model.init_hidden(test_loader.batch_size, self.device)
        with torch.no_grad():
            ''' 
            In the validation loop, the gradients should not be updated. 
            Therefore, ensure that the gradients are not calculated during the forward pass of the valudation.
            To implement this, use torch.no_grad 
            '''
            for batch_X, _ in test_loader:
                batch_X = batch_X.to(self.device) # For processing, move batch tensor and target to device
                batch_predictions, hidden = self.model(batch_X, hidden) # Use the model validation mode to make predictions with the batch_X
                predictions.extend(batch_predictions.cpu().numpy()) # Append current batch predictions (as numpy array from device) to prediction list

        return np.array(predictions)
    

#### Questions:
* Difference between extend and append
* Difference between np. array and np.ndarray

In [230]:
# METHOD TO TRAIN AND EVALUATE 
def train_and_evaluate_eso(
    X_train_eso_scaled: pd.DataFrame, 
    y_train_eso: pd.DataFrame,
    X_test_eso_scaled: pd.DataFrame, 
    y_test_eso: pd.DataFrame,
) -> Tuple[dict, dict]:
    
    # Log initial shapes
    logger.info(f"Initial shapes:")
    logger.info(f"X_train: {X_train_eso_scaled.shape}")
    logger.info(f"y_train: {y_train_eso.shape}")
    logger.info(f"X_test: {X_test_eso_scaled.shape}")
    logger.info(f"y_test: {y_test_eso.shape}")
    
    eso_config = ModelConfig(dataset_type=DatasetType.ESO)
    eso_modeler = Modeler(eso_config)

    logger.info(f'Training ESO model...')
    eso_cv_scores = eso_modeler.cross_validate(X_train_eso_scaled, y_train_eso)

    train_loader, _ = eso_modeler.prepare_data(X_train_eso_scaled, y_train_eso)

    best_val_loss = float('inf')
    patience = 5
    patience_counter = 0

    for epoch in range(eso_config.epochs):
        train_loss = eso_modeler.train_epoch(train_loader)
                                             
        # Early stopping check
        if train_loss < best_val_loss:
            best_val_loss = train_loss
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            logger.info(f'Early stopping at epoch {epoch + 1}')
            break
            
        if (epoch + 1) % 5 == 0:
            logger.info(f'Epoch {epoch + 1}/{eso_config.epochs}, Train Loss: {train_loss:.4f}')                        
    
    eso_predictions = eso_modeler.predict(X_test_eso_scaled)

    # Calculate test metrics
    y_test_values = y_test_eso['FOSSIL'].values
    test_mse = mean_squared_error(y_test_values, eso_predictions)
    test_mae = mean_absolute_error(y_test_values, eso_predictions)
    test_r2 = r2_score(y_test_values, eso_predictions)
    
    logger.info('\nESO Test Set Metrics:')
    logger.info(f'Test MSE: {test_mse:.2f}')
    logger.info(f'Test MAE: {test_mae:.2f}')
    logger.info(f'Test R2: {test_r2:.4f}')

    return {
        'cv_scores': eso_cv_scores,
        'predictions': eso_predictions,
        'test_metrics': {
            'mse': test_mse,
            'mae': test_mae,
            'r2': test_r2
        }
    }


In [231]:
# Train and evaluate ESO model
eso_results = train_and_evaluate_eso(
    X_train_eso_scaled, y_train_eso,
    X_test_eso_scaled, y_test_eso
)

# Access ESO results
eso_predictions = eso_results['predictions']
eso_cv_scores = eso_results['cv_scores']
eso_test_metrics = eso_results['test_metrics']


Fold 1/2


UnboundLocalError: local variable 'hidden' referenced before assignment